In [1]:
import pandas as pd
from pathlib import Path
from datetime import date, timedelta
def find_gaps(sorted_list):
    gaps = {}
    end_of_df = sorted_list[-1]
    gap_id = None
    for i in range(1, len(sorted_list)):
        prev_num = sorted_list[i-1]
        curr_num = sorted_list[i]
        
        # Если разница больше 1, значит есть дырка
        if curr_num - prev_num > 1:
            gap_start = prev_num + 1
            gap_end = curr_num - 1
            
            # Сохраняем в словарь: ключом делаем номер дырки для удобства
            # Можно использовать кортеж (start, end) как ключ, если удобнее
            gap_id = f"gap_{len(gaps) + 1}"
            
            gaps[gap_id] = {
                'start': gap_start,
                'end': gap_end,
                'length': gap_end - gap_start + 1,
                'block': [gap_end+1, end_of_df] # Опционально: список всех пропущенных чисел
            }
        if gap_id is not None:
            gaps[gap_id]['block'][1] = curr_num
        
            
    return gaps

### Данные

In [2]:
current_dir = Path.cwd() 
pi_e_folder = current_dir / 'data/pi_e'

files = [f for f in pi_e_folder.iterdir() if f.is_file()]
files_dict = {}
for file in files:
    fstr = file.name   
    files_dict['20' + fstr.split('.')[0].split('_')[-1]] = fstr

def get_month_range(start_year, start_month, end_year, end_month):
    current = date(start_year, start_month, 1)
    stop = date(end_year, end_month, 1)
    
    months_list = []
    while current <= stop:
        months_list.append(current.strftime('%Y-%m'))
        
        current = (current.replace(day=1) + timedelta(days=32)).replace(day=1)
        
    return months_list
start_y, start_m = 2020, 9   
end_y, end_m = 2026, 2      

months_list = get_month_range(start_y, start_m, end_y, end_m)

from openpyxl import load_workbook

for month in months_list:
    if month not in files_dict:
        continue

    filename = files_dict[month]
    file_path = pi_e_folder / filename
    
    try:
        # Открываем книгу только чтобы посмотреть имена листов
        wb = load_workbook(file_path, read_only=True)
        sheet_names = wb.sheetnames
        wb.close()
        
        # Выбираем нужный лист
        if 'Таблица' in sheet_names:
            sheet_name = 'Таблица'
        elif 'Результат' in sheet_names:
            sheet_name = 'Результат'
        else:
            print(f'⚠️ В файле {filename} нет нужных листов. Доступны: {sheet_names}')
            files_dict[month] = None
            continue
            
        # Читаем данные уже зная имя листа
        files_dict[month] = pd.read_excel(file_path, sheet_name=sheet_name)
        
    except Exception as e:
        print(f'❌ Ошибка обработки файла {month}: {e}')
        files_dict[month] = None

pie_data = {}
for month in months_list:
    month_dict = {} 
    df = files_dict[month]
    if df is not None:
        df = df.set_axis(pd.RangeIndex(0, df.shape[1]), axis = 1)
        if df.iloc[:, 1].loc[['Население' in str(i) and 'в целом' in str(i) for i in df.iloc[:, 1]]].shape[0] != 0:
            start_index = df.iloc[:, 1].loc[['Население' in str(i) and 'в целом' in str(i) for i in df.iloc[:, 1]]].index.start
        elif df.iloc[:, 1].loc[['Все' in str(i) and 'опрошенные' in str(i) for i in df.iloc[:, 1]]].shape[0] != 0:
            start_index = df.iloc[:, 1].loc[['Все' in str(i) and 'опрошенные' in str(i) for i in df.iloc[:, 1]]].index.start
        else:
            print(f'бросил {month}')
            continue
        razrez_raw = df.loc[start_index].copy(deep = True)
        razrez_data = df.loc[start_index+1].copy(deep = True)
        razrez = razrez_raw.unique()[1:]
        razrez_indexes = razrez_raw.loc[[i in razrez for i in razrez_raw]].index.to_list().copy() + [df.shape[1]+1]
        config = {v :{'start': razrez_indexes[i],
                 'stop': razrez_indexes[i+1]-1, 
                 'data': razrez_data[razrez_indexes[i+1]:razrez_indexes[i+2]].to_list()} for i, v in enumerate(razrez[1:])}
        index_level_0 = ['Население в целом'] # Категория (Пол, Возраст...)
        index_level_1 = ['100%'] # Группа (мужчины, женщины...)

        for category, info in config.items():
            count = len(info['data'])
            # Добавляем название категории столько раз, сколько элементов в списке data
            index_level_0.extend([category.replace('-\n', '').replace('-', '')] * count)
            # Добавляем сами названия групп
            index_level_1.extend(info['data'])

        # 3. Создаем MultiIndex и присваиваем его DataFrame
        multi_index = pd.MultiIndex.from_arrays([index_level_0, index_level_1], names=['Категория', 'Группа'])

        index_list = []
        for i, row in df.iloc[start_index+2:].iterrows():
            if row.unique().shape[0] >= 3:
                index_list.append(i)
        gaps = find_gaps(index_list)
        print(f'========== {month} ==========')
        for gap in gaps:
            question = df.iloc[gaps[gap]['end'],0].replace('\xa0', ' ').replace('-\n', '')
            month_dict[question] = df.iloc[gaps[gap]['block'][0]:gaps[gap]['block'][1]+1].set_index(0).rename_axis(None).set_axis(multi_index, axis = 1).astype(float)
        pie_data[month] = month_dict

⚠️ В файле Infl_exp_25-08.xlsx нет нужных листов. Доступны: ['Инфляционные ожидания', 'Потреб. и фин. настроения', 'Сберегательные настроения', 'Эконом. настроения', 'Кредитные настр. и повед.', 'Материальное положение', 'Экономия', 'Ставки', 'О роли БР в сдерж инфл', 'Инфляция', 'Данные для графиков', 'Данные за все годы']
========== 2020-09 ==========
========== 2020-10 ==========
========== 2020-11 ==========
========== 2020-12 ==========
========== 2021-01 ==========
========== 2021-02 ==========
========== 2021-03 ==========
========== 2021-04 ==========
========== 2021-05 ==========
========== 2021-06 ==========
========== 2021-07 ==========
========== 2021-08 ==========
========== 2021-09 ==========
========== 2021-10 ==========
========== 2021-11 ==========
========== 2021-12 ==========
========== 2022-01 ==========
========== 2022-02 ==========
========== 2022-03 ==========
========== 2022-04 ==========
========== 2022-05 ==========
========== 2022-06 ==========
========== 202

In [13]:
sd_df = pd.DataFrame(index= ['есть', 'нет'])
for month in pie_data:
    print(f'============= {month} =============')
    key = None
    for k in pie_data[month]:
        # search by the key word
        if 'расходов были у Вас' in k:
            key = k
            break
        elif 'есть какие-либо кредиты' in k:
            key = k
            break
    
    if key is None:
        print('Не нашел вопроса')

    for k in pie_data[month][key].columns:
        if 'береж' in k[0]:
            category_key = k[0]
            break
    for k in pie_data[month][key][category_key].index:
        if 'нет' in k:
            row_key = k
            break
    print(f'Вопрос: {key}')
    print(f'Категория: {category_key}')
    print(f'Ответ: {row_key}')
        
    sd_df[month] = pie_data[month][key][category_key].loc[row_key, :]

============= 2020-09 =============
Вопрос: Какие из перечисленных на карточке крупных расходов были у Вас (Вашей семьи) за последние три месяца? 
(Карточка, любое число ответов.)
Категория: Наличие сбережений


NameError: name 'row_key' is not defined

In [12]:
['расходов были у Вас' in k for k in list(pie_data['2020-09'].keys())]

[False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False]

In [4]:
sd_df['2025-08'] = (sd_df['2025-07']+sd_df['2025-09'])/2
sd_df.to_excel('data/sd.xlsx')